# Processamento dos dados

Este notebook baixa os World Development Indicators (WDI), do Banco Mundial, e gera os dados do [site](https://wilson-castro.github.io/riqueza-compra-saude/). O Google Colaboratory (Colab) permite executar o notebook no navegador.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilson-castro/riqueza-compra-saude/blob/main/processamento/processamento.ipynb)


In [ ]:
import csv
import json
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

if "google.colab" in sys.modules:
    RAIZ = Path("/content/riqueza-compra-saude")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "https://github.com/wilson-castro/riqueza-compra-saude.git", str(RAIZ)], check=True)
else:
    atual = Path.cwd().resolve()
    RAIZ = next((p for p in (atual, *atual.parents)
                 if (p / "index.html").is_file() and (p / "processamento").is_dir()), None)
    if RAIZ is None:
        raise FileNotFoundError("Execute o notebook dentro do repositório.")

ENTRADAS = RAIZ / "processamento" / "entradas"
BRUTOS = RAIZ / "processamento" / "brutos"
INTERMEDIARIOS = RAIZ / "processamento" / "dados"
SITE_DADOS = RAIZ / "dados"
for pasta in (BRUTOS, INTERMEDIARIOS, SITE_DADOS):
    pasta.mkdir(parents=True, exist_ok=True)


## Download

Esta etapa guarda os arquivos de origem e extrai os três CSV usados pelo processamento. Apague um arquivo em `brutos/` para baixá-lo de novo.


In [ ]:
def baixar(url, destino):
    if not destino.exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino.name}: {destino.stat().st_size:,} bytes")

zip_wdi = BRUTOS / "WDI_CSV.zip"
geojson = BRUTOS / "ne_110m_admin_0_countries.geojson"
baixar("https://databank.worldbank.org/data/download/WDI_CSV.zip", zip_wdi)
baixar("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_110m_admin_0_countries.geojson", geojson)
with zipfile.ZipFile(zip_wdi) as arquivo:
    nomes = {Path(nome).name.lower(): nome for nome in arquivo.namelist() if not nome.endswith("/")}
    for nome in ("WDICSV.csv", "WDICountry.csv", "WDISeries.csv"):
        with arquivo.open(nomes[nome.lower()]) as origem, (BRUTOS / nome).open("wb") as destino:
            shutil.copyfileobj(origem, destino)


## Indicadores

Esta lista define os indicadores, os agregados e os anos do processamento. PPC significa paridade de poder de compra.


In [ ]:
ANOS = range(1990, 2025)

# Cada código do WDI leva à coluna, ao nome em português, à unidade e a se o eixo usa escala log.
INDICADORES = {
    "NY.GDP.PCAP.PP.KD": ("pib_pc_ppp", "PIB per capita, PPC", "US$ internacionais de 2021", True),
    "SP.DYN.LE00.IN": ("expectativa_vida", "Expectativa de vida", "anos", False),
    "SP.POP.TOTL": ("populacao", "População", "habitantes", True),
    "SP.DYN.IMRT.IN": ("mortalidade_infantil", "Mortalidade infantil", "por mil nascidos vivos", True),
    "SP.DYN.TFRT.IN": ("fecundidade", "Fecundidade", "filhos por mulher", False),
    "EG.ELC.ACCS.ZS": ("acesso_eletricidade", "Acesso à eletricidade", "% da população", False),
    "IT.NET.USER.ZS": ("uso_internet", "Uso de internet", "% da população", False),
    "SP.URB.TOTL.IN.ZS": ("populacao_urbana", "População urbana", "% da população", False),
    "NV.AGR.TOTL.ZS": ("agropecuaria_pib", "Agropecuária no PIB", "% do PIB", False),
    "SH.H2O.BASW.ZS": ("agua_potavel", "Água potável básica", "% da população", False),
    "SP.POP.65UP.TO.ZS": ("populacao_65_mais", "População com 65 anos ou mais", "% da população", False),
    "SP.DYN.LE00.FE.IN": ("expectativa_vida_mulheres", "Expectativa de vida das mulheres", "anos", False),
    "SP.DYN.LE00.MA.IN": ("expectativa_vida_homens", "Expectativa de vida dos homens", "anos", False),
    "SL.TLF.CACT.FE.ZS": ("mulheres_trabalho", "Mulheres na força de trabalho",
                          "% das mulheres com 15 anos ou mais", False),
    "SP.ADO.TFRT": ("fecundidade_adolescente", "Fecundidade adolescente",
                    "nascimentos por mil mulheres de 15 a 19 anos", False),
}
# A diferença não existe pronta no WDI. Ela é a expectativa das mulheres menos a dos homens.
DIFERENCA = ("diferenca_vida_genero", "Diferença de expectativa de vida (mulheres menos homens)", "anos")

AGREGADOS = {"WLD", "EAS", "ECS", "LCN", "MEA", "NAC", "SAS", "SSF", "LIC", "LMC", "UMC", "HIC"}


## Tabela completa

Esta etapa reúne os valores por país e ano e grava as tabelas intermediárias.


In [ ]:
origem = BRUTOS
destino = INTERMEDIARIOS
csv.field_size_limit(10**8)


def ler(nome):
    with open(origem / nome, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


paises = {}
for p in ler("WDICountry.csv"):
    codigo = p["Country Code"]
    if p["Region"] or codigo in AGREGADOS:
        paises[codigo] = {"pais": p["Table Name"], "regiao": p["Region"], "renda": p["Income Group"],
                          "agregado": not p["Region"]}

fontes = {s["Series Code"]: s["Source"] for s in ler("WDISeries.csv")}

valores = {}
with open(origem / "WDICSV.csv", encoding="utf-8-sig") as f:
    leitor = csv.reader(f)
    cabecalho = next(leitor)
    posicoes = [(cabecalho.index(str(a)), a) for a in ANOS]
    for linha in leitor:
        iso3, codigo = linha[1], linha[3]
        if iso3 not in paises or codigo not in INDICADORES:
            continue
        coluna = INDICADORES[codigo][0]
        for i, ano in posicoes:
            if linha[i]:
                valores.setdefault((iso3, ano), {})[coluna] = float(linha[i])

for v in valores.values():
    if "expectativa_vida_mulheres" in v and "expectativa_vida_homens" in v:
        v[DIFERENCA[0]] = round(v["expectativa_vida_mulheres"] - v["expectativa_vida_homens"], 3)

colunas = [c for c, *_ in INDICADORES.values()] + [DIFERENCA[0]]
campos = ["iso3", "pais", "regiao", "renda", "ano"] + colunas
arquivos = {False: open(destino / "wdi_completo.csv", "w", newline=""),
            True: open(destino / "wdi_completo_agregados.csv", "w", newline="")}
escritores = {k: csv.DictWriter(f, campos) for k, f in arquivos.items()}
for w in escritores.values():
    w.writeheader()
for iso3 in sorted(paises):
    p = paises[iso3]
    for ano in ANOS:
        escritores[p["agregado"]].writerow({"iso3": iso3, "pais": p["pais"], "regiao": p["regiao"],
                                            "renda": p["renda"], "ano": ano,
                                            **valores.get((iso3, ano), {})})
for f in arquivos.values():
    f.close()

so_paises = [k for k in paises if not paises[k]["agregado"]]
grade = len(so_paises) * len(ANOS)
metadados = [(codigo, c, nome, unidade, log) for codigo, (c, nome, unidade, log) in INDICADORES.items()]
metadados.append(("SP.DYN.LE00.FE.IN menos SP.DYN.LE00.MA.IN", *DIFERENCA, False))
with open(destino / "indicadores.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["coluna", "codigo", "nome", "unidade", "escala_log", "fonte", "cobertura",
                "paises_2023", "ultimo_ano"])
    for codigo, coluna, nome, unidade, log in metadados:
        anos = [a for (iso3, a), v in valores.items() if coluna in v and iso3 in so_paises]
        n2023 = sum(1 for a in anos if a == 2023)
        w.writerow([coluna, codigo, nome, unidade, int(log), fontes.get(codigo, "Banco Mundial (WDI)"),
                    round(len(anos) / grade, 3), n2023, max(anos)])
        print(f"{coluna:28s} cobertura={len(anos) / grade:.0%} países em 2023={n2023}")
print(f"países: {len(so_paises)}, agregados: {len(paises) - len(so_paises)}")


## Dados do site

Esta etapa reduz os valores e grava os CSV usados pelo site.


In [ ]:
ORIGEM = INTERMEDIARIOS
DESTINO = SITE_DADOS

# Códigos do Banco Mundial para regiões e grupos de renda. O site usa o código curto.
REGIAO = {
    "East Asia & Pacific": "EAS",
    "Europe & Central Asia": "ECS",
    "Latin America & Caribbean": "LCN",
    "Middle East & North Africa": "MEA",
    "North America": "NAC",
    "South Asia": "SAS",
    "Sub-Saharan Africa": "SSF",
}
RENDA = {
    "Low income": "LIC",
    "Lower middle income": "LMC",
    "Upper middle income": "UMC",
    "High income": "HIC",
}
# PIB e população viram inteiros. Os outros indicadores ficam com duas casas.
INTEIROS = {"pib_pc_ppp", "populacao"}
# As expectativas por sexo só servem para calcular a diferença, que já vem pronta.
FORA = {"expectativa_vida_mulheres", "expectativa_vida_homens"}


def arredondar(coluna, valor):
    if valor == "":
        return ""
    numero = float(valor)
    if coluna in INTEIROS:
        return str(round(numero))
    texto = f"{numero:.2f}".rstrip("0").rstrip(".")
    return texto if texto != "-0" else "0"


def gravar(caminho, cabecalho, linhas):
    with open(caminho, "w", newline="", encoding="utf-8") as arquivo:
        escritor = csv.writer(arquivo)
        escritor.writerow(cabecalho)
        escritor.writerows(linhas)


def main():
    DESTINO.mkdir(parents=True, exist_ok=True)
    with open(ENTRADAS / "iso2.csv", encoding="utf-8") as arquivo:
        iso2 = {linha["iso3"]: linha["iso2"] for linha in csv.DictReader(arquivo)}
    # As regiões seguem o Gapminder, com a Oceania separada da Ásia.
    # Ilhas do Canal e Kosovo ficam na Europa.
    with open(ENTRADAS / "gapminder_4regioes.csv", encoding="utf-8") as arquivo:
        regiao4 = {linha["iso3"]: linha["regiao4"] for linha in csv.DictReader(arquivo)}
    for iso3 in "ASM AUS FJI FSM GUM KIR MHL MNP NCL NRU NZL PLW PNG PYF SLB TON TUV VUT WSM".split():
        regiao4[iso3] = "oceania"

    with open(ORIGEM / "indicadores.csv", encoding="utf-8") as arquivo:
        indicadores = [linha for linha in csv.DictReader(arquivo) if linha["coluna"] not in FORA]
    colunas = [linha["coluna"] for linha in indicadores]
    gravar(
        DESTINO / "indicadores.csv",
        ["coluna", "codigo", "nome", "unidade", "escala_log", "cobertura", "paises_2023"],
        [[i["coluna"], i["codigo"], i["nome"], i["unidade"], i["escala_log"],
          i["cobertura"], i["paises_2023"]] for i in indicadores],
    )

    paises = {}
    serie = []
    with open(ORIGEM / "wdi_completo.csv", encoding="utf-8") as arquivo:
        for linha in csv.DictReader(arquivo):
            paises[linha["iso3"]] = [
                linha["iso3"], iso2.get(linha["iso3"], ""), linha["pais"],
                REGIAO.get(linha["regiao"], ""), RENDA.get(linha["renda"], ""),
                regiao4.get(linha["iso3"], ""),
            ]
            serie.append([linha["iso3"], linha["ano"]]
                         + [arredondar(c, linha[c]) for c in colunas])
    gravar(DESTINO / "paises.csv", ["iso3", "iso2", "nome", "regiao", "renda", "regiao4"],
           sorted(paises.values()))

    with open(ORIGEM / "wdi_completo_agregados.csv", encoding="utf-8") as arquivo:
        for linha in csv.DictReader(arquivo):
            serie.append([linha["iso3"], linha["ano"]]
                         + [arredondar(c, linha[c]) for c in colunas])
    gravar(DESTINO / "serie.csv", ["iso3", "ano"] + colunas, serie)

    print(f"{len(paises)} países e {len(serie)} linhas gravados em {DESTINO}")


main()


## Mapa

Esta etapa grava os contornos do Natural Earth com os códigos dos países do Banco Mundial.


In [ ]:
PAISES = SITE_DADOS / "paises.csv"
SAIDA = SITE_DADOS / "mundo.json"

# O Natural Earth usa KOS para Kosovo, e o Banco Mundial usa XKX.
TROCAS = {"KOS": "XKX"}


def codigo(propriedades, do_banco):
    # O código ISO falta em alguns países, como a França, e vem como -99. Nesse caso vale o
    # próximo campo que tiver um código conhecido pelo Banco Mundial.
    for campo in ("ISO_A3", "ISO_A3_EH", "ADM0_A3", "WB_A3"):
        valor = TROCAS.get(propriedades.get(campo), propriedades.get(campo))
        if valor in do_banco:
            return valor
    return None


def arredondar(coordenadas):
    if isinstance(coordenadas[0], (int, float)):
        return [round(coordenadas[0], 2), round(coordenadas[1], 2)]
    return [arredondar(c) for c in coordenadas]


def main(caminho):
    origem = json.loads(Path(caminho).read_text())
    do_banco = {linha["iso3"] for linha in csv.DictReader(PAISES.open())}
    formas = []
    for f in origem["features"]:
        if f["properties"].get("ADM0_A3") == "ATA":
            continue
        formas.append({
            "type": "Feature",
            "id": codigo(f["properties"], do_banco),
            "properties": {},
            "geometry": {"type": f["geometry"]["type"], "coordinates": arredondar(f["geometry"]["coordinates"])},
        })
    SAIDA.write_text(json.dumps({"type": "FeatureCollection", "features": formas}, separators=(",", ":")))
    no_mapa = {f["id"] for f in formas if f["id"]}
    print(f"{len(formas)} formas gravadas em {SAIDA}. {len(no_mapa)} países do Banco Mundial estão no mapa.")


main(geojson)


## Conferência

Esta etapa mostra a cobertura dos indicadores e compara o produto interno bruto (PIB) per capita em PPC com a expectativa de vida no ano mais recente com dados para ambos.


In [ ]:
import matplotlib.pyplot as plt

with (SITE_DADOS / "indicadores.csv").open(encoding="utf-8") as arquivo:
    for indicador in csv.DictReader(arquivo):
        print(f"{indicador['nome']}: cobertura {float(indicador['cobertura']):.1%}, países em 2023 {indicador['paises_2023']}")
with (SITE_DADOS / "paises.csv").open(encoding="utf-8") as arquivo:
    regioes = {p["iso3"]: p["regiao4"] for p in csv.DictReader(arquivo)}
print(f"Países: {len(regioes)}")
with (SITE_DADOS / "serie.csv").open(encoding="utf-8") as arquivo:
    pontos = [linha for linha in csv.DictReader(arquivo)
              if linha["iso3"] in regioes and linha["pib_pc_ppp"] and linha["expectativa_vida"]
              and float(linha["pib_pc_ppp"]) > 0]
ano = max(int(p["ano"]) for p in pontos)
cores = {"africa": "#e15759", "americas": "#4e79a7", "asia": "#f28e2b", "europe": "#59a14f", "oceania": "#b07aa1", "": "#888888"}
fig, ax = plt.subplots()
for regiao in sorted({regioes[p["iso3"]] for p in pontos if int(p["ano"]) == ano}):
    grupo = [p for p in pontos if int(p["ano"]) == ano and regioes[p["iso3"]] == regiao]
    ax.scatter([float(p["pib_pc_ppp"]) for p in grupo],
               [float(p["expectativa_vida"]) for p in grupo],
               color=cores.get(regiao, "#888888"), label=regiao or "Sem região", alpha=0.7)
ax.set_xscale("log")
ax.set_xlabel("PIB per capita, PPC")
ax.set_ylabel("Expectativa de vida em anos")
ax.set_title(str(ano))
ax.legend(title="Região")
plt.show()


## Publicação

No Colab, a célula abaixo junta a pasta `dados/` num arquivo zip e oferece o download. Troque os arquivos de `dados/` do repositório pelos do zip e publique o site. No computador, os arquivos já ficam em `dados/` e basta publicar.


In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files

    pacote = shutil.make_archive("/content/dados_site", "zip", SITE_DADOS)
    files.download(pacote)
